# Video Captioning Example Notebook

This notebook demonstrates how to use the video captioning system implemented in the `VideoCaptioner.py` module. In this example, we configure a `GPT4oCaptioner` and run it on a sample video. Make sure to adjust the `video_path` variable to point to an actual video file on your system.

In [1]:
# Add the 'src' directory to the system path
import sys
import os

current_dir = os.path.dirname(os.path.abspath('__file__'))
src_path = os.path.join(current_dir, 'SurgicalFeedbackGeneration', 'src')
if src_path not in sys.path:
    sys.path.append(src_path)

# Import the necessary classes from VideoCaptioner.py
from src.VideoCaptioner import VideoCaptionerConfig, GPT4oCaptioner
from src.VideoProcessor import VideoProcessor
from utils import set_envvars

set_envvars()

print('Imports successful!')

Imports successful!


In [2]:
# Configure the captioner
config = VideoCaptionerConfig(
    openai_api_key=os.environ["OPENAI_API_KEY"],
    min_frame_similarity=0.85,
    max_tokens_per_frame_generation=100,
    target_fps=10
)

# Create a GPT4oCaptioner instance
captioner = GPT4oCaptioner(config)

print('Captioner configured successfully!')

Captioner configured successfully!


In [3]:
clips_dir = '/home/firdavs/surgery/clips_no_wiggle/fb_clips_no_wiggle'
clip_path = os.path.join(clips_dir, 'c5_s0_1-12-53.avi')
mp4_path = 'data/tmp/temp_video.mp4'

VideoProcessor.convert_avi_to_mp4(clip_path, mp4_path.strip(".mp4"), overwrite=True)

True

In [5]:
from IPython.display import Video

Video(mp4_path)

In [6]:
import cv2
import numpy as np

def load_video(video_path: str) -> np.ndarray:
    # Load video file
    video = cv2.VideoCapture(video_path)

    # Get video properties
    frame_width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = video.get(cv2.CAP_PROP_FPS)

    # Initialize list to store frames
    frames = []

    # Read frames from video
    while True:
        ret, frame = video.read()
        if not ret:
            break
        frames.append(frame)

    # Convert to numpy array
    frames = np.array(frames)

    # Release video object
    video.release()
    
    return frames

def get_metrics(frames: np.ndarray) -> dict:
    num_frames = len(frames)
    height, width, _ = frames[0].shape
    print(f"num_frames: {num_frames}, height: {height}, width: {width}")

get_metrics(load_video(mp4_path))

num_frames: 52, height: 250, width: 320


In [7]:
sample_context = {
    "system": "You are a medical expert specialized in urology procedures.",
    "user": "The video clip shows a urology surgery, specifically a nephrectomy procedure. Provide a concise and medically accurate description of the scene."
}

try:
    caption_output = captioner.caption_video(mp4_path, sample_context)
    print('Video Caption Output:')
    print(caption_output)
except Exception as e:
    print('An error occurred while captioning the video:')
    print(e)

Video Caption Output:


In [15]:
combined_caption = caption_output.get_combined_caption()
frame_captions = caption_output.get_frame_captions()
for i, frame_caption in enumerate(frame_captions):
    print(f"Frame {i}: {frame_caption}")


Frame 0: The image shows a nephrectomy in progress, with surgical instruments working to dissect and remove the kidney from surrounding tissues.
Frame 1: The image shows surgeons using surgical instruments to dissect and remove a kidney during a nephrectomy procedure.
Frame 2: The image shows surgical instruments being used to dissect and remove kidney tissue during a nephrectomy procedure.
Frame 3: The surgical team is performing a nephrectomy, utilizing laparoscopic instruments to carefully excise the kidney from surrounding tissue and structures.
Frame 4: The image shows surgical instruments being used to carefully dissect and remove renal tissue during a nephrectomy.
Frame 5: The image depicts a surgical field where a nephrectomy is being performed, with surgical instruments visible maneuvering around kidney tissue to facilitate its removal.
Frame 6: The image shows a surgical scene of a nephrectomy where instruments are being used to dissect and remove the kidney tissue.
Frame 7: 

In [17]:
print(f"Combined caption: {combined_caption}")

Combined caption: The video clip captures a nephrectomy procedure, where a surgical team, employing both traditional and robotic-assisted techniques, meticulously dissects and removes the kidney. Utilizing a range of instruments, they carefully isolate the kidney from surrounding tissues, aiming to manage vascular structures and control bleeding throughout the process.


In [9]:
import pickle
pickle.dump(caption_output, open('caption_output.pkl', 'wb'))